# Lezione 6 — Pratica: il problema XOR

Questa pratica riprende il circuito della lezione teorica e ti propone di rimuoverne un componente per verificare con i tuoi occhi perché quel componente è indispensabile.

## Mettilo in pratica: elimina il `CNOT` iniziale

Nella lezione teorica hai visto che il circuito inizia con `circuito.append(cirq.CNOT(qubits[0], qubits[1]))`, una porta che crea l'entanglement tra i due qubit di ingresso prima di applicare qualsiasi rotazione addestrabile. La spiegazione teorica dice che questo `CNOT` iniziale è la fonte della non linearità che permette di risolvere lo XOR.

Il tuo compito è verificarlo sperimentalmente:

1. Copia `construir_circuito` ed elimina da essa la riga del `CNOT` iniziale (lascia il resto esattamente uguale)
2. Riaddestra il modello da zero con le stesse 512 epoche e gli stessi dati dello XOR
3. Disegna di nuovo la frontiera di decisione con il nuovo modello
4. Confronta l'accuratezza finale (`precision_hinge`) del modello senza `CNOT` iniziale con quella del modello originale
5. Annota quale differenza osservi e mettila in relazione con il ruolo dell'entanglement: il circuito senza `CNOT` iniziale si comporta come due classificatori indipendenti, esattamente come un perceptron classico?

**Suggerimenti**

- Riutilizza `codificar_dato`, `generar_datos_xor`, `rotacion_de_un_qubit`, `agrupamiento_dos_qubits` e `precision_hinge` così come sono: non cambiano
- Devi modificare solo il corpo di `construir_circuito`
- Usa un nome di variabile diverso per il nuovo modello (ad esempio `modelo_sin_entrelazamiento`) per non perdere il modello originale e poter confrontare entrambi

In [ ]:
import tensorflow_quantum as tfq
import tensorflow as tf
import cirq
import sympy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


def codificar_dato(qubits, circuito, dato):
    """X^i = Rx(pi * i): permite codificar valores continuos entre 0 y 1."""
    circuito.append([cirq.X(qubits[0]) ** dato[0]])
    circuito.append([cirq.X(qubits[1]) ** dato[1]])
    return circuito


def generar_datos_xor(qubits):
    datos = [[0, 0], [1, 0], [0, 1], [1, 1]]
    etiquetas = [-1, 1, 1, -1]
    circuitos = [codificar_dato(qubits, cirq.Circuit(), d) for d in datos]
    return tfq.convert_to_tensor(circuitos), np.array(etiquetas)


def rotacion_de_un_qubit(qubit, simbolos):
    return cirq.Circuit(
        cirq.X(qubit) ** simbolos[0],
        cirq.Y(qubit) ** simbolos[1],
        cirq.Z(qubit) ** simbolos[2],
    )


def agrupamiento_dos_qubits(fuente, receptor, simbolos):
    """Combina la información de dos qubits en la respuesta de uno solo (el receptor)."""
    circuito = cirq.Circuit()
    selector_receptor = rotacion_de_un_qubit(receptor, simbolos[0:3])
    selector_fuente = rotacion_de_un_qubit(fuente, simbolos[3:6])
    circuito.append(selector_receptor)
    circuito.append(selector_fuente)
    circuito.append(cirq.CNOT(control=fuente, target=receptor))
    circuito.append(selector_receptor ** -1)
    return circuito


def precision_hinge(etiqueta_real, prediccion):
    etiqueta_real = tf.squeeze(etiqueta_real) > 0.0
    prediccion = tf.squeeze(prediccion) > 0.0
    return tf.reduce_mean(tf.cast(etiqueta_real == prediccion, tf.float32))


qubits = [cirq.GridQubit(0, i) for i in range(2)]
entrenamiento, etiquetas = generar_datos_xor(qubits)

In [ ]:
def construir_circuito_sin_entrelazamiento(qubits):
    """Igual que construir_circuito, pero sin el CNOT inicial que entrelaza los qubits de entrada."""
    x1, y1, z1, x2, y2, z2 = sympy.symbols("x1 y1 z1 x2 y2 z2")
    agrupamiento = sympy.symbols("agrupamiento0:6")
    circuito = cirq.Circuit()
    # TODO: no añadas aquí el CNOT inicial entre qubits[0] y qubits[1]
    circuito.append(cirq.rx(x1).on(qubits[0]))
    circuito.append(cirq.ry(y1).on(qubits[0]))
    circuito.append(cirq.rz(z1).on(qubits[0]))
    circuito.append(cirq.rx(x2).on(qubits[1]))
    circuito.append(cirq.ry(y2).on(qubits[1]))
    circuito.append(cirq.rz(z2).on(qubits[1]))
    circuito += agrupamiento_dos_qubits(qubits[0], qubits[1], agrupamiento)
    return circuito


# TODO: comprueba con un print(construir_circuito_sin_entrelazamiento(qubits)) que el CNOT inicial ya no aparece

In [ ]:
# TODO: monta el modelo de Keras con tfq.layers.PQC, igual que en la clase de teoría,
# pero usando construir_circuito_sin_entrelazamiento en vez de construir_circuito.
# Llama a la variable del modelo "modelo_sin_entrelazamiento" para no confundirla con el modelo original.

operador_lectura = [cirq.Z(qubits[1])]
entradas = tf.keras.Input(shape=(), dtype=tf.dtypes.string)
capa_cuantica = tfq.layers.PQC(
    # TODO: sustituye None por construir_circuito_sin_entrelazamiento(qubits)
    None, operador_lectura, repetitions=32,
    differentiator=tfq.differentiators.ParameterShift(),
    initializer=tf.keras.initializers.Zeros,
)(entradas)
modelo_sin_entrelazamiento = tf.keras.models.Model(inputs=entradas, outputs=capa_cuantica)
modelo_sin_entrelazamiento.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                                    loss=tf.keras.losses.hinge, metrics=[precision_hinge])

In [ ]:
# TODO: reentrena desde cero con las mismas 512 épocas y los mismos datos del XOR
historial_sin_entrelazamiento = None  # sustituye None por la llamada a .fit(...)

In [ ]:
# TODO: vuelve a dibujar la frontera de decisión, ahora con modelo_sin_entrelazamiento,
# barriendo el mismo plano [-0.1, 1.1] x [-0.1, 1.1] que en la clase de teoría.

mapa_colores = ListedColormap(["midnightblue", "maroon"])
N = 150
x1_malla, x2_malla = np.meshgrid(np.linspace(-0.1, 1.1, N), np.linspace(-0.1, 1.1, N))
puntos = np.c_[x1_malla.ravel(), x2_malla.ravel()]

# TODO: sustituye "modelo_sin_entrelazamiento" si has usado otro nombre de variable
predicciones = [None for p in puntos]  # TODO: calcula la predicción de cada punto
predicciones = np.asarray(predicciones).reshape(x1_malla.shape)

plt.pcolormesh(x1_malla, x2_malla, predicciones, cmap=mapa_colores)
plt.scatter([0, 1], [0, 1], color="blue", label="-1")
plt.scatter([0, 1], [1, 0], color="red", label="1")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

# TODO: compara historial.history["val_precision_hinge"][-1] (modelo original)
# con historial_sin_entrelazamiento.history["val_precision_hinge"][-1] (este modelo)

<details>
<summary>💡 Mostra soluzione</summary>

```python
def construir_circuito_sin_entrelazamiento(qubits):
    """Come construir_circuito, ma senza il CNOT iniziale che crea l'entanglement tra i qubit di ingresso."""
    x1, y1, z1, x2, y2, z2 = sympy.symbols("x1 y1 z1 x2 y2 z2")
    agrupamiento = sympy.symbols("agrupamiento0:6")
    circuito = cirq.Circuit()
    # Senza il CNOT iniziale: i due qubit non sono più in entanglement prima delle rotazioni
    circuito.append(cirq.rx(x1).on(qubits[0]))
    circuito.append(cirq.ry(y1).on(qubits[0]))
    circuito.append(cirq.rz(z1).on(qubits[0]))
    circuito.append(cirq.rx(x2).on(qubits[1]))
    circuito.append(cirq.ry(y2).on(qubits[1]))
    circuito.append(cirq.rz(z2).on(qubits[1]))
    circuito += agrupamiento_dos_qubits(qubits[0], qubits[1], agrupamiento)
    return circuito


operador_lectura = [cirq.Z(qubits[1])]
entradas = tf.keras.Input(shape=(), dtype=tf.dtypes.string)
capa_cuantica = tfq.layers.PQC(
    construir_circuito_sin_entrelazamiento(qubits), operador_lectura, repetitions=32,
    differentiator=tfq.differentiators.ParameterShift(),
    initializer=tf.keras.initializers.Zeros,
)(entradas)
modelo_sin_entrelazamiento = tf.keras.models.Model(inputs=entradas, outputs=capa_cuantica)
modelo_sin_entrelazamiento.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
                                    loss=tf.keras.losses.hinge, metrics=[precision_hinge])

historial_sin_entrelazamiento = modelo_sin_entrelazamiento.fit(
    entrenamiento, etiquetas, epochs=512, batch_size=4,
    validation_data=(entrenamiento, etiquetas),
)

mapa_colores = ListedColormap(["midnightblue", "maroon"])
N = 150
x1_malla, x2_malla = np.meshgrid(np.linspace(-0.1, 1.1, N), np.linspace(-0.1, 1.1, N))
puntos = np.c_[x1_malla.ravel(), x2_malla.ravel()]
predicciones = [
    modelo_sin_entrelazamiento(
        tfq.convert_to_tensor([codificar_dato(qubits, cirq.Circuit(), p)])
    ).numpy().flatten()[0]
    for p in puntos
]
predicciones = np.asarray(predicciones).reshape(x1_malla.shape)

plt.pcolormesh(x1_malla, x2_malla, predicciones, cmap=mapa_colores)
plt.scatter([0, 1], [0, 1], color="blue", label="-1")
plt.scatter([0, 1], [1, 0], color="red", label="1")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

precision_original = historial.history["val_precision_hinge"][-1]
precision_sin_entrelazamiento = historial_sin_entrelazamiento.history["val_precision_hinge"][-1]
print("Accuratezza con CNOT iniziale:", precision_original)
print("Accuratezza senza CNOT iniziale:", precision_sin_entrelazamiento)
```

Senza il `CNOT` iniziale, il circuito tratta ciascun qubit di ingresso separatamente: la rotazione di `qubits[0]` dipende solo dal primo bit di ingresso, e quella di `qubits[1]` solo dal secondo. Il raggruppamento finale rimescola i due qubit, ma ormai è troppo tardi, perché l'informazione in ingresso non è mai stata combinata prima di quel rimescolamento. Il risultato è un modello che si comporta come due classificatori indipendenti, esattamente come un perceptron classico, e la cui accuratezza finale resta ferma intorno al 50%: il circuito non distingue le quattro combinazioni dello XOR, come prevede la spiegazione teorica sulla necessità dell'entanglement.

</details>